## Imports & src definitions

All `src/` logic (UMF, User, create_dataloader, decentralized_train_n_epochs, etc.)
is inlined here so the notebook is fully self-contained.

In [25]:
from torch.utils.data import Dataset, DataLoader
import torch
import torch.nn as nn
from torch.optim import SGD
from tqdm.notebook import tqdm
from sklearn.model_selection import train_test_split
import pandas as pd
import numpy as np
import networkx as nx
import time
import math
import copy
import os
SEED = 42
torch.manual_seed(0)
np.random.seed(0)

In [27]:
# ── UMF model (mirrors src/models/MatrixFactorization.py UMF class) ──────────
# UMF = User Matrix Factorization.
# Each user owns ONE model: only item embeddings Q (shared via gradients),
# plus a single user vector p (local, never shared).
class UMF(nn.Module):
    def __init__(self, n_items: int, n_factors: int = 30, sparse: bool = False):
        super().__init__()
        self.Q = nn.Embedding(n_items, n_factors, sparse=sparse)   # item embeddings
        self.p = nn.Parameter(torch.zeros(n_factors))              # this user's vector
        nn.init.normal_(self.Q.weight, std=0.01)
        nn.init.normal_(self.p,        std=0.01)

    def forward(self, item_ids: torch.Tensor) -> torch.Tensor:
        # item_ids: (bs,)  ->  score: (bs,)
        q = self.Q(item_ids)           # (bs, n_factors)
        return (self.p * q).sum(dim=-1)  # inner product


# ── User container (mirrors src/users.py) ────────────────────────────────────
class User:
    def __init__(self, id: int, model: nn.Module,
                 optimizer, model_name: str = "umf"):
        self.id         = id
        self.model      = model
        self.optimizer  = optimizer
        self.model_name = model_name


print("UMF and User defined.")

UMF and User defined.


In [29]:
# ── DataLoader (mirrors src/data_utils.py create_dataloader with dl_type='rs')
# 'rs' = random-sampling: each __getitem__ returns a random batch of
# batch_size interactions for user at position idx.
class RSDataset(Dataset):
    """Random-sampling dataset: one item per user per call to __getitem__."""
    def __init__(self, df: pd.DataFrame, batch_size: int = 10,
                 user_col: str = "customer_id",
                 item_col: str = "product_code",
                 rating_col: str = "bought"):
        self.batch_size = batch_size
        users   = torch.tensor(df[user_col].values,   dtype=torch.long)
        items   = torch.tensor(df[item_col].values,   dtype=torch.long)
        ratings = torch.tensor(df[rating_col].values, dtype=torch.float32)

        unique_users = users.unique()
        self.user_ids = unique_users
        self.user_to_indices = {
            u.item(): (users == u).nonzero(as_tuple=True)[0]
            for u in unique_users
        }
        self.items   = items
        self.ratings = ratings

    def __len__(self):
        return len(self.user_ids)

    def __getitem__(self, idx):
        uid     = self.user_ids[idx].item()
        indices = self.user_to_indices[uid]
        bs      = min(len(indices), self.batch_size)
        sel     = indices[torch.randperm(len(indices))[:bs]]
        # Return (user_idx, item_idx) tensor and ratings — same shape as ML-100K
        x = torch.stack([
            torch.full((bs,), uid, dtype=torch.long),
            self.items[sel]
        ], dim=1)  # (bs, 2): col0=user, col1=item
        y = self.ratings[sel]  # (bs,)
        return x, y


def create_dataloader(df: pd.DataFrame, dl_type: str = "rs",
                      batch_size: int = 10, p=None,
                      user_col: str = "customer_id",
                      item_col: str = "product_code",
                      rating_col: str = "bought") -> DataLoader:
    """Mirrors src/data_utils.py create_dataloader for dl_type='rs'."""
    assert dl_type == "rs", f"Only 'rs' loader type is implemented here; got '{dl_type}'"
    dataset = RSDataset(df, batch_size=batch_size,
                        user_col=user_col, item_col=item_col, rating_col=rating_col)
    return DataLoader(dataset, batch_size=1, shuffle=(batch_size > 1))


print("DataLoader defined.")

DataLoader defined.


In [31]:
# ── Graph builder (mirrors src/graphs.py random_k_out_graph) ─────────────────
def random_k_out_graph(n: int, k: int = 2, seed: int = 1) -> nx.DiGraph:
    """Directed random k-out graph with preferential attachment (alpha=0.5)."""
    return nx.random_k_out_graph(n, k=k, alpha=0.5, self_loops=False, seed=seed)


print("Graph builder defined.")

Graph builder defined.


In [33]:
# ── Gradient sharing (FOAF Algorithm A5) ─────────────────────────────────────
# Only ∇Q (item embedding gradients) are shared; ∇p stays local.
BYTES_PER_PARAM = 4  # float32

def share_gradient(sender: User, user_models: dict,
                   graph: nx.DiGraph,
                   item_ids=None):
    """
    Push sender's ∇Q to out-neighbours.  Neighbours apply it immediately.
    Returns (n_commutes, commute_cost_bytes).
    """
    neighbors = list(graph.successors(sender.id))
    if not neighbors:
        return 0, 0

    Q_grad = sender.model.Q.weight.grad
    if Q_grad is None:
        return 0, 0

    lr = sender.optimizer.param_groups[0]["lr"]

    if item_ids is not None:
        unique_items = item_ids.unique()
        grad_slice   = Q_grad[unique_items].detach()
        n_params     = grad_slice.numel()
        for nid in neighbors:
            with torch.no_grad():
                user_models[nid].model.Q.weight.data[unique_items] -= lr * grad_slice
    else:
        grad_slice = Q_grad.detach()
        n_params   = grad_slice.numel()
        for nid in neighbors:
            with torch.no_grad():
                user_models[nid].model.Q.weight.data -= lr * grad_slice

    return len(neighbors), n_params * BYTES_PER_PARAM * len(neighbors)


print("share_gradient defined.")

share_gradient defined.


In [35]:
# ── Training functions (mirrors src/training/decentralized.py) ────────────────

def decentralized_validate_loop(user_models: dict,
                                 loader: DataLoader) -> float:
    """Returns RMSE across all (user, item) pairs in loader."""
    for _, u in user_models.items():
        u.model.eval()
    sq_sum, n_obs = 0.0, 0
    with torch.no_grad():
        for x, y in loader:
            if x.ndim == 3:
                x, y = x.squeeze(0), y.squeeze(0)
            uid   = int(x[0, 0])
            score = user_models[uid].model(x[:, 1])
            sq_sum += ((score - y) ** 2).sum().item()
            n_obs  += x.shape[0]
    return math.sqrt(sq_sum / max(n_obs, 1))


def _train_one_epoch(user_models: dict, train_loader: DataLoader,
                     graph: nx.DiGraph) -> tuple:
    """One epoch: returns (avg_train_loss, total_commutes, total_commute_cost)."""
    loss_fn = nn.MSELoss(reduction="mean")
    for _, u in user_models.items():
        u.model.train()

    total_sq, total_n = 0.0, 0
    total_commutes, total_cost = 0, 0

    pbar = tqdm(train_loader, leave=False)
    for x, y in pbar:
        if x.ndim == 3:
            x, y = x.squeeze(0), y.squeeze(0)

        uid      = int(x[0, 0])
        item_ids = x[:, 1]
        user     = user_models[uid]

        user.optimizer.zero_grad()
        score = user.model(item_ids)
        loss  = loss_fn(score, y)
        loss.backward()

        # Update own parameters first, then share ∇Q
        user.optimizer.step()
        n_comm, cost = share_gradient(user, user_models, graph, item_ids=item_ids)

        total_commutes += n_comm
        total_cost     += cost
        total_sq       += loss.item() * x.shape[0]
        total_n        += x.shape[0]

    train_rmse = math.sqrt(total_sq / max(total_n, 1))
    return train_rmse, total_commutes, total_cost


def decentralized_train_n_epochs(
    user_models: dict,
    train_loader: DataLoader,
    val_loader: DataLoader,
    epochs: int = 50,
    graph: nx.DiGraph = None,
    patience: int = 3,
) -> tuple:
    """
    Train for up to `epochs` rounds with early stopping.
    Returns (train_losses, val_losses, time_per_epoch, commutes)
    where commutes = {'number_of_commutes': [...], 'commute_cost': [...]}
    """
    train_losses, val_losses, time_per_epoch = [], [], []
    commute_counts, commute_costs = [], []

    best_val   = float("inf")
    best_state = None
    p_count    = 0

    for epoch in range(1, epochs + 1):
        t0 = time.time()
        train_rmse, n_comm, cost = _train_one_epoch(user_models, train_loader, graph)
        val_rmse   = decentralized_validate_loop(user_models, val_loader)
        elapsed    = time.time() - t0

        train_losses.append(train_rmse)
        val_losses.append(val_rmse)
        time_per_epoch.append(elapsed)
        commute_counts.append(n_comm)
        commute_costs.append(cost)

        print(f"Epoch {epoch} | Train Loss: {train_rmse:.4f} "
              f"| Validation Loss: {val_rmse:.4f} "
              f"| Time Elapsed: {elapsed:.6f} sec "
              f"|Commute: {n_comm} "
              f"| Commute Cost: {cost}")

        if val_rmse < best_val:
            best_val   = val_rmse
            best_state = {uid: copy.deepcopy(u.model.state_dict())
                          for uid, u in user_models.items()}
            p_count = 0
        else:
            p_count += 1
            if p_count >= patience:
                print("Early stopping.")
                break

    print(f"\nTotal time elapsed: {sum(time_per_epoch)}")

    # Restore best model
    if best_state is not None:
        for uid, u in user_models.items():
            u.model.load_state_dict(best_state[uid])

    commutes = {"number_of_commutes": commute_counts,
                "commute_cost":       commute_costs}
    return train_losses, val_losses, time_per_epoch, commutes


print("Training functions defined.")

Training functions defined.


## Parameter Setting

In [38]:
model_type       = "umf"
val_loader_type  = "rs"
train_loader_type = "rs"
userprop         = None
n_factors        = 30
sparse           = False
batch_size       = 10
lr               = 0.03871364416669273
weight_decay     = 0.14214480688557163
mom              = 0.001
graph_seed       = 1
n_epochs         = 50
SEED             = 0

## Main

In [49]:
cache_tag  = f"u{TARGET_USERS}_m{MIN_TRAIN_INTERACTIONS}_s{SEED}"
train_path = os.path.join(SAMPLED_DATA_DIR, f"train_{cache_tag}.csv")
train_df = pd.read_csv(train_path)

FileNotFoundError: [Errno 2] No such file or directory: 'hm_sampled/train_u500_m20_s0.csv'

In [51]:
from pathlib import Path

cwd = Path.cwd()
print(cwd)

/Users/haowen/Documents/Decentral RS/rebuttal/code


In [53]:
# ── Load sampled H&M dataset from cache ───────────────────────────────────────
# Set these to match what was used when sampling was run.
TARGET_USERS           = 500
MIN_TRAIN_INTERACTIONS = 20
SAMPLED_DATA_DIR       = "./hm_sampled"

cache_tag  = f"u{TARGET_USERS}_m{MIN_TRAIN_INTERACTIONS}_s{SEED}"
train_path = os.path.join(SAMPLED_DATA_DIR, f"train_{cache_tag}.csv")
val_path   = os.path.join(SAMPLED_DATA_DIR, f"val_{cache_tag}.csv")
test_path  = os.path.join(SAMPLED_DATA_DIR, f"test_{cache_tag}.csv")
meta_path  = os.path.join(SAMPLED_DATA_DIR, f"meta_{cache_tag}.csv")

assert all(os.path.exists(p) for p in [train_path, val_path, test_path, meta_path]), (
    f"Cached files not found in '{SAMPLED_DATA_DIR}/'.\n"
    "Run the hm_foaf_experiment_sampled.ipynb preprocessing section first "
    "to generate the cache."
)

train_df = pd.read_csv(train_path)
val_df   = pd.read_csv(val_path)
test_df  = pd.read_csv(test_path)
meta_df  = pd.read_csv(meta_path)

n_users = int(meta_df.loc[meta_df["key"] == "n_users", "value"].iloc[0])
n_items = int(meta_df.loc[meta_df["key"] == "n_items", "value"].iloc[0])

print(f"Loaded from cache: {cache_tag}")
print(f"Total User: {n_users}")
print(f"Total Item: {n_items}")
train_df.head()

AssertionError: Cached files not found in './hm_sampled/'.
Run the hm_foaf_experiment_sampled.ipynb preprocessing section first to generate the cache.

In [ ]:
# ── DataLoaders ───────────────────────────────────────────────────────────────
train_data_loader = create_dataloader(
    df=train_df, dl_type=train_loader_type, batch_size=batch_size, p=userprop
)
val_data_loader  = create_dataloader(df=val_df,  dl_type=val_loader_type)
test_data_loader = create_dataloader(df=test_df, dl_type=val_loader_type)

print(f"Train batches: {len(train_data_loader)} | "
      f"Val batches: {len(val_data_loader)} | "
      f"Test batches: {len(test_data_loader)}")

In [ ]:
# ── Initialise one UMF model per user ─────────────────────────────────────────
users = {}
for i in tqdm(range(n_users)):
    user_model = UMF(n_items, n_factors=n_factors, sparse=sparse)
    users[i] = User(
        id=i,
        model=user_model,
        optimizer=SGD(user_model.parameters(),
                      lr=lr, weight_decay=weight_decay, momentum=mom),
        model_name=model_type,
    )

In [ ]:
# ── Build graph ───────────────────────────────────────────────────────────────
graph = random_k_out_graph(n=n_users, k=2, seed=graph_seed)

In [ ]:
# ── Train ─────────────────────────────────────────────────────────────────────
torch.manual_seed(0)
train_losses, val_losses, time_per_epoch, commutes = decentralized_train_n_epochs(
    user_models=users,
    train_loader=train_data_loader,
    val_loader=val_data_loader,
    epochs=n_epochs,
    graph=graph,
)

In [ ]:
# ── Test evaluation ───────────────────────────────────────────────────────────
test_loss = decentralized_validate_loop(users, test_data_loader)

In [ ]:
test_loss

## Save Results

In [ ]:
import pickle
from pathlib import Path

out_dir = Path("results")
out_dir.mkdir(exist_ok=True)

tag = f"HM_rs_random_2_out_{cache_tag}"

# ── Full pickle (preserves commutes dict exactly) ─────────────────────────────
result = {
    "train_losses":    train_losses,
    "val_losses":      val_losses,
    "time_per_epoch":  time_per_epoch,
    "commutes":        commutes,
    "test_loss":       test_loss,
    "n_users":         n_users,
    "n_items":         n_items,
    "cache_tag":       cache_tag,
}
with open(out_dir / f"{tag}.pkl", "wb") as f:
    pickle.dump(result, f)

# ── Per-epoch CSV (val RMSE + cumulative comm cost) ───────────────────────────
cum_mb = np.cumsum(commutes["commute_cost"]) / 1e6
pd.DataFrame({
    "epoch":      range(1, len(val_losses) + 1),
    "train_rmse": train_losses,
    "val_rmse":   val_losses,
    "time_sec":   time_per_epoch,
    "commutes":   commutes["number_of_commutes"],
    "comm_mb":    cum_mb,
}).to_csv(out_dir / f"{tag}_per_epoch.csv", index=False)

print(f"Test RMSE  : {test_loss:.4f}")
print(f"Best val   : {min(val_losses):.4f}")
print(f"Total comm : {cum_mb[-1]:.1f} MB")
print(f"Saved to   : results/{tag}.*")